In [0]:
# ============================================================
# CHESS LIVE ANALYTICS PROJECT
# SILVER LAYER CONFIGURATION
# ============================================================

BRONZE_TABLE = "grandmasters.bronze.tv_feed"

SILVER_MOVES = "grandmasters.silver.game_move_events"

SILVER_MOVES_CHECKPOINT_LOCATION = "/Volumes/grandmasters/silver/checkpoints/game_move_events"

spark.sql("""
          CREATE VOLUME IF NOT EXISTS grandmasters.silver.checkpoints
        COMMENT 'Checkpoint storage for streaming jobs';
          """)

In [0]:
bronze_stream = spark.readStream.format("delta").table(BRONZE_TABLE)

In [0]:
# ============================================================
# EXTRACT MOVE EVENTS
#
# Purpose:
# Parse every FEN event into a structured move timeline.
# One row represents one move in a chess game.
# ============================================================

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

fen_schema = StructType([
    StructField("t", StringType()),
    StructField("d", StructType([
        StructField("fen", StringType()),
        StructField("lm", StringType()),
        StructField("wc", IntegerType()),
        StructField("bc", IntegerType())
    ]))
])

move_events_stream_df = (
    bronze_stream
    .filter(col("event_type") == "fen")
    .withColumn(
        "json",
        from_json(col("raw_event"), fen_schema)
    )
    .select(
        "game_id",
        "move_seq",
        "ingested_at",
        col("json.d.fen").alias("fen"),
        col("json.d.lm").alias("last_move"),
        col("json.d.wc").alias("white_clock_secs"),
        col("json.d.bc").alias("black_clock_secs")
    )
)

query = (
    move_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", SILVER_MOVES_CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(SILVER_MOVES)
)

query.awaitTermination()

print(f"Silver batch complete for move_events. Processed up to latest available bronze offset.")